# Stage 2 — FDE Extraction Pipeline
### Local LLM · Gemma 4 · HuggingFace Transformers

This notebook walks through every step of the extraction pipeline so you can
test and inspect each function independently before running the full loop.

**Run order:** execute cells top-to-bottom. Cells 1–13 define all functions
and run inline tests; cell 14 is the full pipeline loop.

| # | Section |
|---|---------|
| 1 | Imports & Logging |
| 2 | Configuration (paths, model, generation settings) |
| 3 | Pydantic Schema |
| 4 | Codebook System Prompt |
| 5 | Stage 1 Loader & Filter |
| 6 | Article Text Loader |
| 7 | Model Loader *(slow — ~GB download on first run)* |
| 8 | Prompt Builder |
| 9 | Inference |
| 10 | JSON Extractor |
| 11 | Pydantic Validator |
| 12 | CSV Row Builder |
| 13 | Output Writers |
| 14 | ▶ Full Pipeline Loop |
| 15 | Results Inspector |


In [1]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/gdrive')

# Change working directory to your file position
data_folder = '/content/gdrive/My Drive/WWF_2026/data/'
os.chdir(data_folder)

# Confirm the change
print("Working Directory:", os.getcwd())

Mounted at /content/gdrive
Working Directory: /content/gdrive/My Drive/WWF_2026/data


In [2]:
extraction_folder = 'n_extracted'
extraction_folder_path = os.path.join(os.getcwd(), extraction_folder)

dispute_detection_file = 'Gemma4_e4b_summary_2.json'
stage1_path = os.path.join(os.getcwd(), 'n_dispute_detection', dispute_detection_file)

In [3]:
# Example positive output
# from pprint import pprint
# import json
# fde_output = """{"article_id": "nDOC_012", "fde_criteria": {"min_two_actors": true, "discrete_temporal_moment": true, "discrete_location": true, "fisheries_resource_contested": true}, "date": "2009-03-15", "year": 2009, "country": "Tanzania", "region": 1, "location": "Off Tanzanian Coast", "loc_accuracy": 2, "actor1": "Tanzania Government/Joint Operation", "actor2": "Flagless foreign trawler/crew", "ViolenceScore": 2, "drivers": {"WeakGov": false, "FishPop": false, "EcoChngOther": false, "Poverty": false, "FoodInsecurity": false, "Marginalization": false, "GroundsLim": true, "OpsScales": false, "ForeignFisher": true, "Markets": false, "IncrEfficiency": false, "IncrPressure": false, "Crime": true, "StratLoc": false}, "issue_note": "The event involved the seizure and arrest of a foreign trawler operating illegally in Tanzania's EEZ, resulting in the confiscation of 70 tonnes of tuna fish. This constitutes physical action (arrest/seizure) and is driven by the presence of foreign fishers and illegal fishing (piracy)."}"""
# json_obj = json.loads(fde_output)
# print(json.dumps(json_obj, indent=2))

In [6]:
# Example negative output
# raw_output_content = """<|channel>thought
# Thinking Process:

# 1.  **Analyze the Request:** The goal is to extract all distinct Fisheries Dispute Events (FDEs) from the provided news article (nDOC_008) and return them as a strict JSON array, adhering to all defined schemas and rules.

# 2.  **Analyze the Article (nDOC_008):**
#     *   **Topic:** Blast fishing (dynamite fishing) in Tanzania.
#     *   **Core Conflict:** Illegal use of explosives by fishermen vs. the law/environmental protection/other fishermen.
#     *   **Key Elements:**
#         *   Practice: Blast fishing (using explosives/fertilizers).
#         *   Location: Coast of Tanzania, Zanzibar, Mtwara to Tanga.
#         *   Actors: Fishermen using explosives (perpetrators) vs. Law enforcement/Community (victims/reporting parties).
#         *   Time: Ongoing, increasing in the past five years (since 2009/2010).
#         *   Consequences: Coral destruction, reduced fish stocks, injuries (missing limbs), lucrative business.
#         *   Enforcement issues: Difficulty tracking, bribery, threats of retribution.

# 3.  **Identify Potential FDEs (Discrete Events):**
#     *   The article describes a widespread, ongoing *practice* (blast fishing) and systemic issues (weak enforcement, high profit, environmental damage).
#     *   *FDE Definition Check:* An FDE requires a *discrete temporal moment* and a *discrete location*.
#     *   The article mentions:
#         1.  "The explosives fishermen use are usually made from readily available cheap craft components... They are dropped overboard off the coast of Tanzania and Zanzibar..." (General practice/location).
#         2.  "Another horrific consequence is the frequent sight of fishermen in Tanzania walking around with missing limbs." (General consequence/location).
#         3.  "Local fishermen have informed the police about those who use the crude fishing method, but it has not stemmed the practice." (Specific interaction/event).
#         4.  "They say that those arrested pay a bribe and come back, and if they find out who reported them they threaten retribution." (Specific interaction/event).

# 4.  **Determine if Discrete Events Exist:**
#     *   The article focuses heavily on the *systemic problem* (illegal practice, weak governance, environmental decline). While specific incidents (arrests, injuries) are mentioned, they are presented as generalized examples of the ongoing crisis, not as single, reportable, discrete events with specific dates/locations/actors in conflict.
#     *   *Example:* "Fishermen walking around with missing limbs" is a consequence, not a single FDE.
#     *   *Example:* "Local fishermen have informed the police" is a general statement about a recurring pattern, not a single, dated event.

# 5.  **Conclusion on FDE Extraction:** The text describes a chronic, widespread conflict and its consequences, but it does not provide the necessary granularity (specific date, specific location, specific incident) to qualify as a distinct FDE under the strict definitions.

# 6.  **Final Output Generation:** Since no distinct FDEs meeting all criteria are found, the output must be an empty JSON array. (Rule: If no FDEs are found, return []).<channel|>[]<turn|>"""

# # Escape the raw_output_content for JSON
# escaped_raw_output = json.dumps(raw_output_content)

# # Construct the full JSON string using f-string, embedding the escaped raw_output
# fde_fail_output = f'{{"article_id": "nDOC_008", "error": "json_parse_failed", "raw_output": {escaped_raw_output}}}'

# json_obj = json.loads(fde_fail_output)
# print(json.dumps(json_obj, indent=2))

## 1 · Imports & Logging

In [7]:
import csv
import json
import logging
import sys
import textwrap
import time
from datetime import datetime
from pathlib import Path
from typing import Literal, Optional

import torch
from accelerate import Accelerator
from pydantic import BaseModel, Field, ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline

# Logging setup — outputs to stdout so Jupyter shows it inline
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print("Imports OK")
print(f"  torch version  : {torch.__version__}")
print(f"  CUDA available : {torch.cuda.is_available()}")


Imports OK
  torch version  : 2.11.0+cu128
  CUDA available : True


## 2 · Configuration

Edit paths and model settings here. Every downstream cell reads from these
variables — this is the **only** cell you need to change between runs.


In [8]:
# ── Paths ─────────────────────────────────────────────────────────────────
STAGE1_PATH  = Path(stage1_path)   # Stage 1 detection JSON
ARTICLES_DIR = Path(f"{extraction_folder_path}/")            # Folder of {article_id}.txt files
OUTPUT_DIR   = Path("../results/")             # JSONL + CSV + failed log land here

# ── Model ──────────────────────────────────────────────────────────────────
# HuggingFace model ID,
# "Qwen/Qwen3.5-9B" models have long thinking output
# "meta-llama/Llama-3.2-3B-Instruct" greedy, not much reasoning
MODEL_NAME    = "google/gemma-4-E4B-it"
DEVICE        = "auto"                      # "auto" | "cpu" | "cuda" | "cuda:0"

# ── Generation ─────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 2048    # Increase for very long articles
TEMPERATURE    = 0.1     # Near-deterministic; raise to ~0.3 for more variety

# ── Run control ────────────────────────────────────────────────────────────
LIMIT = 3    # Set to an int (e.g. 3) to process only the first N articles

# ── Output filenames (auto-timestamped) ────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
JSONL_PATH  = OUTPUT_DIR / f"fde_events_{run_ts}.jsonl"
CSV_PATH    = OUTPUT_DIR / f"fde_events_{run_ts}.csv"
FAILED_PATH = OUTPUT_DIR / f"fde_failed_{run_ts}.jsonl"

print(f"Stage 1 input : {STAGE1_PATH}")
print(f"Articles dir  : {ARTICLES_DIR}")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"Model         : {MODEL_NAME}")
print(f"JSONL output  : {JSONL_PATH}")
print(f"CSV output    : {CSV_PATH}")
print(f"Failed log    : {FAILED_PATH}")


Stage 1 input : /content/gdrive/My Drive/WWF_2026/data/n_dispute_detection/Gemma4_e4b_summary_2.json
Articles dir  : /content/gdrive/My Drive/WWF_2026/data/n_extracted
Output dir    : ../results
Model         : google/gemma-4-E4B-it
JSONL output  : ../results/fde_events_20260610_200259.jsonl
CSV output    : ../results/fde_events_20260610_200259.csv
Failed log    : ../results/fde_failed_20260610_200259.jsonl


## 3 · Pydantic Schema

Three nested models defining the expected structure of every extracted FDE.
Pydantic validates the model's JSON output against these before anything is
written to disk. A smoke-test at the end confirms the schema parses correctly.


In [9]:
class FdeCriteria(BaseModel):
    min_two_actors: bool = Field(description="At least two actors are involved")
    discrete_temporal_moment: bool = Field(
        description="Event has a discrete time period (not merely systemic)"
    )
    discrete_location: bool = Field(
        description="Event has a bounded, identifiable location"
    )
    fisheries_resource_contested: bool = Field(
        description="A fisheries resource is contested/disputed/source of conflict"
    )


class ConflictDrivers(BaseModel):
    WeakGov: bool
    FishPop: bool
    EcoChngOther: bool
    GroundsLim: bool
    ForeignFisher: bool
    IllegalFishing: bool
    IncrEfficiency: bool
    OpsScales: bool
    Markets: bool
    Poverty: bool
    FoodInsecurity: bool
    Marginalization: bool
    StratLoc: bool
    MaritimeCrime: bool


class FdeEvent(BaseModel):
    article_id: str
    fde_criteria: FdeCriteria
    date: str = Field(description="Date FDE began in YYYY-MM-DD. Best estimate if uncertain.")
    year: int = Field(description="Year of event")
    country: str = Field(description="Country in which the FDE occurred")
    region: Literal[0, 1, 2, 3, 4] = Field(
        description="0=Arctic 1=Horn_of_Africa 2=Oceania 3=South_America 4=South_Asia"
    )
    location: str = Field(description="Closest identifiable land-based location")
    loc_accuracy: Literal[0, 1, 2, 3, 4, 5] = Field(
        description="0=Country 1=District 2=International 3=NA 4=Region 5=Town"
    )
    actor1: str = Field(description="Primary actor")
    actor2: str = Field(description="Secondary actor")
    ViolenceScore: int = Field(
        ge=1, le=3,
        description=(
            "1=Verbal/complaints/bans/fines  "
            "2=Property_damage/confiscation/arrests/abductions  "
            "3=Injuries/assaults/fatalities"
        ),
    )
    drivers: ConflictDrivers
    issue_note: str = Field(
        description=(
            "1-2 sentences summarising the event AND justifying key coding choices "
            "(ViolenceScore, actors, any true drivers)."
        )
    )


# ── Smoke test ────────────────────────────────────────────────────────────
_sample = {
    "article_id": "TEST",
    "fde_criteria": {
        "min_two_actors": True, "discrete_temporal_moment": True,
        "discrete_location": True, "fisheries_resource_contested": True,
    },
    "date": "2010-01-01", "year": 2010, "country": "Tanzania",
    "region": 1, "location": "Dar es Salaam", "loc_accuracy": 4,
    "actor1": "Blast fishers", "actor2": "Tanzanian police",
    "ViolenceScore": 2,
    "drivers": {k: False for k in ConflictDrivers.model_fields},
    "issue_note": "Smoke-test event.",
}
_ev = FdeEvent(**_sample)
print(f"Schema OK  |  {_ev.country}, ViolenceScore={_ev.ViolenceScore}")


Schema OK  |  Tanzania, ViolenceScore=2


## 4 · Codebook System Prompt

The instruction block injected as the `system` role in every model call.
Distilled from `Fisheries_Conflict_Codebook`.

Edit the prompt text here — all coding rules, region codes, and driver
definitions live in this one cell. No need to touch the pipeline logic.


In [10]:
few_shot = '''FEW-SHOT EXAMPLE

The example below shows the reasoning depth expected for FDE drivers.

* ARTICLE (abbreviated):
  "Tanzania has woken up to confront pirate fishing in its waters. Last
  week's arrest of a flagless foreign trawler some 100 miles off the
  Tanzanian coast by a South African-led joint operation revealed what
  had been happening for many years. The vessel, Tawaqal 1, carried
  70 tonnes of tuna worth an estimated $900,000. Authorities stated that
  more than 70 foreign ships fish illegally in Tanzania's EEZ, depriving
  the country of millions of dollars in marine resources. Reports from
  the Rufiji River Delta described thousands of fish discarded by trawlers
  searching for high-priced species. The government acknowledged that
  illegal foreign ships had prevented Tanzania from benefiting from its
  marine resources and amended laws concerning licensing, fees, and
  penalties."

* CORRECT CODING:
    WeakGov=true
    — "Tanzania may have been losing billions of dollars from foreign
    pirate fishing vessels..." and "over 70 such ships fish illegally
    in its EEZ" indicate longstanding inability to effectively monitor
    and enforce fisheries regulations despite the scale of violations.
    The subsequent legal reforms and regional patrol agreements imply
    previous governance weaknesses.
    FishPop=true
    — "thousands of various species of fish floating in the waters...
    discarded by trawlers" and "denying the country millions of dollars
    worth in marine resources" indicate depletion and loss of fish
    resources associated with illegal exploitation.
    EcoChngOther=true
    — The article reports that trawlers discarded large quantities of
    unwanted fish in Tanzanian waters. The issue note additionally
    specifies that Captain Hsu Ching Tai was charged with polluting the
    environment by dumping fish remains and ship fuel, representing
    environmental degradation beyond simple stock depletion.
    GroundsLim=true
    — The vessel was operating illegally within Tanzania's Exclusive
    Economic Zone (EEZ), violating access and licensing restrictions
    governing who may fish in those waters.
    ForeignFisher=true
    — The central actors are foreign fishers operating aboard a
    "flagless foreign trawler." The crew consisted of Chinese,
    Filipinos, Kenyans, and Vietnamese nationals, and the captain was
    identified as a Chinese national from Taiwan.
    IllegalFishing=true
    — The vessel is repeatedly described as a "pirate ship" engaged in
    "illegal and unregulated fishing." Authorities seized the vessel,
    prosecuted its captain, and noted that dozens of similar ships
    operate illegally in Tanzanian waters.
    IncrEfficiency=false
    — No evidence of advanced or unusually efficient harvesting
    technologies driving increased catch efficiency. The vessel is an
    industrial trawler, but efficiency-enhancing innovation is not
    identified as a causal driver.
    OpsScales=false
    — Although industrial foreign vessels are involved, the article does
    not frame the event as direct competition or conflict between
    industrial and artisanal fishers. The emphasis is on illegal access
    and state losses.
    Markets=false
    — The reference to "high-priced fish species" describes catch
    selection but does not identify market demand, price changes, or
    trade pressures as an explicit driver of the dispute event.
    Poverty=false
    — No indication that poverty among participants motivated the illegal
    fishing activities.
    FoodInsecurity=false
    — Although fish are described as expensive for ordinary Tanzanians,
    food scarcity or insecurity is not presented as a cause of the
    conflict.
    Marginalization=false
    — No evidence of ethnic, social, political, or economic exclusion of
    a particular group contributing to the dispute.
    StratLoc=false
    — The EEZ location is relevant jurisdictionally, but the article does
    not identify control of a strategic maritime chokepoint, boundary,
    or geographically strategic area as a driver.
    MaritimeCrime=false
    — Despite the label "pirate fishing," the event concerns illegal,
    unreported, and unregulated fishing rather than separate maritime
    crimes such as armed piracy, smuggling, kidnapping, or trafficking.'''

In [12]:
SYSTEM_PROMPT = textwrap.dedent("""<|think|>
You are an expert coder for the Fisheries Conflict Database.
Read the supplied news article and extract every distinct Fisheries Dispute
Event (FDE) it contains. Return ONLY a valid JSON array -- no markdown fences,
no commentary. If no FDEs are found, return [].

DEFINITIONS
FDE: An incident in which a fisheries resource is contested, disputed, or the
source of conflict between two or more ACTORS, at a DISCRETE TEMPORAL MOMENT,
in a DISCRETE LOCATION.
  Actor: human individuals or groups (spontaneous or organized). For example:
        Domestic Fishers, Other, Foreign Fishers, Federal Government, Bandits,
        Military, Police, Regional Government, Resource Security, Local Government.
  Discrete temporal moment: days/weeks, NOT a chronic systemic condition.
  Discrete location: a bounded site (village, reef, landing), NOT a diffuse region.

REGION CODES
0 = Arctic
1 = Horn of Africa (Tanzania, Kenya, Somalia, Djibouti, Eritrea, Yemen, Ethiopia,
    Sudan, Uganda, Rwanda, Burundi, DRC, Lake Victoria basin, Red Sea,
    Indian Ocean off East Africa)
2 = Oceania
3 = South America
4 = South Asia

LOC_ACCURACY CODES
0 = Country only
1 = District / Province
2 = International / High-seas
3 = NA (indeterminate)
4 = Region (named but no district/town)
5 = Town / village / named site

VIOLENCE SCORE  (assign the HIGHEST level met)
1 = Verbal only -- bans, complaints, fines, policy disputes
2 = Physical action, NO bodily harm -- property damage, gear/boat confiscation,
    arrests, abductions
3 = Bodily harm -- injuries, sexual assault, fatalities

CONFLICT DRIVERS
A driver is TRUE when the article contains text that explicitly states OR
clearly implies the factor shaped an actor's behaviour. Background context
that frames why the conflict is occurring counts as explicit support.
Do not require a direct causal sentence — pattern-match against the signals
listed below for each driver.
  WeakGov
    TRUE when article mentions: illegal fishing going unchecked / "unabated";
    lack of patrol vessels or funding; enforcement ended after donor program
    closed; authorities unable or underfunded to respond; practice "out of
    control again"; special operation needed to restore order.
    → Any description of enforcement failure or capacity gap = WeakGov=true.
  FishPop
    TRUE when article mentions: fish stocks declining or depleted; fishers
    moving to new grounds because fish are scarce; reduced catches; species
    under pressure; dynamite/destructive fishing described as depleting stocks.
    → Destructive fishing that article links to stock depletion = FishPop=true.
  EcoChngOther
    TRUE when article mentions: coral reef damage or destruction; dynamite
    blasts destroying coral or breeding grounds; pollution harming habitat;
    habitat degradation beyond simple stock decline.
    → Blast fishing destroying "coral rocks / breeding grounds" = EcoChngOther=true.
  Poverty
    TRUE when article explicitly links poverty or economic desperation to why
    actors fish illegally. Mere mention of artisanal fishers is NOT sufficient.
  FoodInsecurity
    TRUE when article explicitly states food access or subsistence need drives
    an actor's behaviour.
  Marginalization
    TRUE when article mentions ethnic, gender, political, or social exclusion
    shaping an actor.
  GroundsLim
    TRUE when article mentions: EEZ violations; foreign vessels in territorial
    waters; fishing bans; restricted/closed areas; illegal access to grounds.
    → Foreign trawlers inside national EEZ or banned waters = GroundsLim=true.
  OpsScales
    TRUE ONLY when the article explicitly contrasts industrial/commercial fleet
    behaviour against artisanal/subsistence fishers AND frames this contrast
    as a driver of the conflict. Do NOT set true simply because both types
    of fishers are mentioned, or because a large vessel is involved.
    → Foreign industrial trawlers vs. local artisanal fishers WITH explicit
      competition framing = OpsScales=true. Trawler arrest alone ≠ OpsScales.
  ForeignFisher
    TRUE when article mentions: foreign-flagged vessels; vessels from named
    foreign countries; "foreign trawlers"; IUU (illegal, unreported, unregulated)
    vessels from outside the nation's EEZ.
  Markets
    TRUE when article explicitly links international demand, export markets, or
    transnational supply chains to the conflict.
  IncrEfficiency
    TRUE when article mentions actors using destructive or highly efficient
    gear (dynamite, poison, fine-mesh nets, sonar) as a driver of conflict.
    → Blast fishing = IncrEfficiency=true in virtually all cases.
  MaritimeCrime
    TRUE ONLY for non-fisheries crime: piracy, kidnapping for ransom,
    drug/arms smuggling, robbery, kidnapping, abduction for ransom, or
    human trafficking at sea. Illegal fishing itself is NOT MaritimeCrime.
  StratLoc
    TRUE when article explicitly discusses the strategic land/shore value of
    the location (military, transit, rebel use) — not the fish.

CODING RULES
1. Code ONLY from information explicitly in the article.
2. Each distinct event (different date OR location OR actors) = a separate FDE.
3. Aggregate/trend/systemic/widespread issue without a discrete event is determined as not FDE,
   stop processing and note evidence(s) in the issue_note.
4. date: YYYY-MM-DD; use -01 placeholders for unknown month/day. NOT article's report date.
5. actor1 = initiating/primary party; actor2 = responding/secondary party.
6. If FDE, evidence standard: a driver is true if the article states it OR if the
   article contains context that directly implies it (e.g. "fishing unabated
   for five years despite ban" implies WeakGov without using that word).
7. issue_note: for EVERY true driver, 1-2 sentences summarising the event AND justifying
 key coding choices (ViolenceScore, actors, any true drivers).
 This is mandatory — do not assert a driver without citing evidence from the text.
8. Return only the JSON array — no extra text before or after.

OUTPUT SCHEMA (one object per FDE)
{
  "article_id": "<string>",
  "fde_criteria": {
    "min_two_actors": <bool>,
    "discrete_temporal_moment": <bool>,
    "discrete_location": <bool>,
    "fisheries_resource_contested": <bool>
  },
  "date": "<YYYY-MM-DD>",
  "year": <int>,
  "country": "<string>",
  "region": <0|1|2|3|4>,
  "location": "<string>",
  "loc_accuracy": <0|1|2|3|4|5>,
  "actor1": "<string>",
  "actor2": "<string>",
  "ViolenceScore": <1|2|3>,
  "drivers": {
    "WeakGov": <bool>, "FishPop": <bool>, "EcoChngOther": <bool>,
    "GroundsLim": <bool>, "ForeignFisher": <bool>, "IllegalFishing": <bool>,
    "IncrEfficiency": <bool>, "OpsScales": <bool>, "Markets": <bool>,
    "Poverty": <bool>, "FoodInsecurity": <bool>, "Marginalization": <bool>,
    "StratLoc": <bool>, "MaritimeCrime": <bool>
  },
  "issue_note": "<string>"
}
""").strip()

print(f"System prompt loaded  |  {len(SYSTEM_PROMPT)} chars  |  "
      f"{len(SYSTEM_PROMPT.splitlines())} lines")


System prompt loaded  |  6878 chars  |  142 lines


In [13]:
'''
WeakGov
  TRUE when article mentions: illegal fishing going unchecked / "unabated";
  lack of patrol vessels or funding; enforcement ended after donor program
  closed; authorities unable or underfunded to respond; practice "out of
  control again"; special operation needed to restore order.
  → Any description of enforcement failure or capacity gap = WeakGov=true.

FishPop
  TRUE when article mentions: fish stocks declining or depleted; fishers
  moving to new grounds because fish are scarce; reduced catches; species
  under pressure; dynamite/destructive fishing described as depleting stocks.
  → Destructive fishing that article links to stock depletion = FishPop=true.

EcoChngOther
  TRUE when article mentions: coral reef damage or destruction; dynamite
  blasts destroying coral or breeding grounds; pollution harming habitat;
  habitat degradation beyond simple stock decline.
  → Blast fishing destroying "coral rocks / breeding grounds" = EcoChngOther=true.

Poverty
  TRUE when article explicitly links poverty or economic desperation to why
  actors fish illegally. Mere mention of artisanal fishers is NOT sufficient.

FoodInsecurity
  TRUE when article explicitly states food access or subsistence need drives
  an actor's behaviour.

Marginalization
  TRUE when article mentions ethnic, gender, political, or social exclusion
  shaping an actor.

GroundsLim
  TRUE when article mentions: EEZ violations; foreign vessels in territorial
  waters; fishing bans; restricted/closed areas; illegal access to grounds.
  → Foreign trawlers inside national EEZ or banned waters = GroundsLim=true.

OpsScales
  TRUE ONLY when the article explicitly contrasts industrial/commercial fleet
  behaviour against artisanal/subsistence fishers AND frames this contrast
  as a driver of the conflict. Do NOT set true simply because both types
  of fishers are mentioned, or because a large vessel is involved.
  → Foreign industrial trawlers vs. local artisanal fishers WITH explicit
    competition framing = OpsScales=true. Trawler arrest alone ≠ OpsScales.

ForeignFisher
  TRUE when article mentions: foreign-flagged vessels; vessels from named
  foreign countries; "foreign trawlers"; IUU (illegal, unreported, unregulated)
  vessels from outside the nation's EEZ.

Markets
  TRUE when article explicitly links international demand, export markets, or
  transnational supply chains to the conflict.

IncrEfficiency
  TRUE when article mentions actors using destructive or highly efficient
  gear (dynamite, poison, fine-mesh nets, sonar) as a driver of conflict.
  → Blast fishing = IncrEfficiency=true in virtually all cases.

MaritimeCrime
  TRUE ONLY for non-fisheries crime: piracy, kidnapping for ransom,
  drug/arms smuggling, robbery, kidnapping, abduction for ransom, or
  human trafficking at sea. Illegal fishing itself is NOT MaritimeCrime.

StratLoc
  TRUE when article explicitly discusses the strategic land/shore value of
  the location (military, transit, rebel use) — not the fish.
'''

'\nWeakGov\n  TRUE when article mentions: illegal fishing going unchecked / "unabated";\n  lack of patrol vessels or funding; enforcement ended after donor program\n  closed; authorities unable or underfunded to respond; practice "out of\n  control again"; special operation needed to restore order.\n  → Any description of enforcement failure or capacity gap = WeakGov=true.\n\nFishPop\n  TRUE when article mentions: fish stocks declining or depleted; fishers\n  moving to new grounds because fish are scarce; reduced catches; species\n  under pressure; dynamite/destructive fishing described as depleting stocks.\n  → Destructive fishing that article links to stock depletion = FishPop=true.\n\nEcoChngOther\n  TRUE when article mentions: coral reef damage or destruction; dynamite\n  blasts destroying coral or breeding grounds; pollution harming habitat;\n  habitat degradation beyond simple stock decline.\n  → Blast fishing destroying "coral rocks / breeding grounds" = EcoChngOther=true.\n\nPo

## 5 · Stage 1 Loader & Filter

Reads the Stage 1 JSON and returns only the `contains_fde=True` articles.

> **Known data quirk in your Stage 1 output:** 18 articles use `"detection"`
> and 2 use `"detected"` as the key name. `get_detections()` handles both.


In [14]:
def get_detections(entry: dict) -> list[dict]:
    """Return the detections list regardless of key spelling (detection / detected)."""
    if "detection" in entry:
        return entry["detection"] or []
    if "detected" in entry:
        return entry["detected"] or []
    log.warning(f"Article {entry.get('article_id', '?')} has no detection/detected key.")
    return []


def is_fde_positive(entry: dict) -> bool:
    """True if ANY detection record has contains_fde=True."""
    return any(d.get("contains_fde", False) for d in get_detections(entry))


def load_and_filter_stage1(path: Path) -> tuple[list[dict], list[dict]]:
    """
    Load Stage 1 JSON. Returns (all_entries, fde_positive_entries).
    Logs key-name inconsistencies and a per-article confidence summary.
    """
    with open(path) as f:
        data: list[dict] = json.load(f)

    # Audit key names and warn if mixed
    key_counts: dict[str, int] = {}
    for entry in data:
        key = ("detection" if "detection" in entry
               else "detected" if "detected" in entry
               else "MISSING")
        key_counts[key] = key_counts.get(key, 0) + 1
    if len(key_counts) > 1:
        log.warning(f"Mixed Stage 1 key names: {key_counts} -- both accepted.")
    else:
        log.info(f"Stage 1 key name: {key_counts}")

    fde_pos = [e for e in data if is_fde_positive(e)]

    log.info(f"Total articles       : {len(data)}")
    log.info(f"contains_fde = True  : {len(fde_pos)}")
    for e in fde_pos:
        dets = get_detections(e)
        conf = next((d.get("confidence") for d in dets if d.get("contains_fde")), None)
        n    = next((d.get("num_candidate_events") for d in dets if d.get("contains_fde")), None)
        log.info(f"  -> {e['article_id']}  confidence={conf}  candidate_events={n}")

    return data, fde_pos


# ── Run ───────────────────────────────────────────────────────────────────
stage1_all, fde_articles = load_and_filter_stage1(STAGE1_PATH)
print(f"\nFDE-positive IDs: {[e['article_id'] for e in fde_articles]}")



FDE-positive IDs: ['nDOC_008', 'nDOC_012']


## 6 · Article Text Loader

Loads a single article `.txt` file by `article_id`.
Returns the text string, or `None` if the file is missing
(the pipeline logs and skips rather than crashing).


In [15]:
def load_article_text(article_id: str, articles_dir: Path) -> Optional[str]:
    """Read {articles_dir}/{article_id}_ext.json. Returns None if not found."""
    path = articles_dir / f"{article_id}_ext.json"
    if not path.exists():
        log.warning(f"Article file not found: {path}")
        return None
    return path.read_text(encoding="utf-8", errors="replace").strip()


# ── Test ─────────────────────────────────────────────────────────────────
_test_id = fde_articles[0]["article_id"] if fde_articles else "nDOC_012"
_text    = load_article_text(_test_id, ARTICLES_DIR)

if _text:
    print(f"Loaded {_test_id}  |  {len(_text)} chars")
    print("--- First 300 chars ---")
    print(_text[:300])
else:
    print(f"  {_test_id}.txt not found in {ARTICLES_DIR}")
    print("  Update ARTICLES_DIR in cell 2 to point at your article files.")


Loaded nDOC_008  |  2967 chars
--- First 300 chars ---
{
    "doc_id": "nDOC_008",
    "title": "Tanzania blast fishing getting out of control",
    "source": "The East African",
    "load_date": "2014-09-13",
    "article_text": "\n\n\nTanzania blast fishing getting out of control\nThe East African\nSeptember 13, 2014 Saturday\n\n\nCopyright 2014 The E


## 7 · Model Loader

Downloads (or loads from `~/.cache/huggingface`) the Gemma 4 model and
wraps it in a HuggingFace `text-generation` pipeline.

> **This cell is slow on first run** (several GB download).
> Once `pipe` is in memory, all downstream cells reuse it — do not re-run
> this cell unnecessarily.


In [16]:
def load_model(model_name: str, device: str):
    """Load tokenizer + model and return a HF text-generation pipeline."""
    log.info(f"Loading tokenizer : {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    log.info(f"Loading model     : {model_name}  device={device}")
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    accelerator = Accelerator()

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=dtype,
        device_map="auto" if device == "auto" else device,
        low_cpu_mem_usage=True,
    )
    model = accelerator.prepare(model)
    model.eval()

    pipe = hf_pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device_map="auto" if device == "auto" else None,
    )
    log.info("Model ready.")
    return pipe


# ── Load (guard against accidental double-load) ───────────────────────────
if "pipe" not in dir() or pipe is None:
    pipe = load_model(MODEL_NAME, DEVICE)
else:
    print("Reusing existing pipe -- skipping reload.")

print(f"Model loaded: {MODEL_NAME}")


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded: google/gemma-4-E4B-it


## 8 · Prompt Builder

Formats the system prompt + article text into a chat-formatted string using
the tokenizer's built-in `apply_chat_template` (works for Gemma-IT, Llama-
Instruct, Qwen-Chat). Falls back to Gemma's native `<start_of_turn>` format
if no chat template is registered.

Run the test block to inspect the exact string sent to the model and the
token count — useful for catching context-length overflows before inference.


In [17]:
def build_prompt(tokenizer, article_id: str, article_text: str) -> str:
    """
    Build the full chat prompt. Uses the tokenizer chat template when available,
    otherwise falls back to Gemma native turn markers.
    """
    user_content = (
        f"ARTICLE ID: {article_id}\n\n"
        f"{article_text.strip()}\n\n"
        "Extract all FDE events from this article and return them as a JSON array."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=True
            )
        except Exception as e:
            log.warning(f"apply_chat_template failed ({e}) -- using fallback")

    # Gemma native fallback
    return (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n{user_content}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )


# ── Test: inspect prompt length for the first FDE article ─────────────────
if _text:
    _prompt = build_prompt(pipe.tokenizer, _test_id, _text)
    _token_count = pipe.tokenizer(_prompt, return_tensors="pt")["input_ids"].shape[-1]
    print(f"Prompt built for {_test_id}")
    print(f"  Tokens : {_token_count}")
    print(f"  Chars  : {len(_prompt)}")
    print("\n--- First 400 chars ---")
    print(_prompt[:400])
    print("\n--- Last 200 chars ---")
    print(_prompt[-200:])
else:
    _prompt = build_prompt(pipe.tokenizer, "TEST", "Placeholder article text.")
    print(f"No article loaded -- dummy prompt ({len(_prompt)} chars):")
    print(_prompt[:300])


Prompt built for nDOC_008
  Tokens : 2425
  Chars  : 10012

--- First 400 chars ---
<bos><|turn>system
<|think|>
<|think|>
You are an expert coder for the Fisheries Conflict Database.
Read the supplied news article and extract every distinct Fisheries Dispute
Event (FDE) it contains. Return ONLY a valid JSON array -- no markdown fences,
no commentary. If no FDEs are found, return [].

DEFINITIONS
FDE: An incident in which a fisheries resource is contested, disputed, or the
source

--- Last 200 chars ---
 a catch of up to 400kg and a profit of $1,800.\n\nLoad-Date: September 13, 2014\n\n\nEnd of Document"
}

Extract all FDE events from this article and return them as a JSON array.<turn|>
<|turn>model



## 9 · Inference

Runs the model on the prompt from cell above and returns only the newly
generated tokens (the input prompt is stripped from the output).

Test this on a single article to verify the model is responding correctly
before committing to the full loop.


In [18]:
def run_inference(pipe, prompt: str,
                  max_new_tokens: int = 2048,
                  temperature: float = 0.1) -> str:
    """Generate model output and return only the new tokens as a plain string."""
    outputs = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        # do_sample=temperature > 0,
        return_full_text=False,          # strip the echoed prompt
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    return outputs[0]["generated_text"].strip()


# ── Test on the first FDE article ─────────────────────────────────────────
if _text:
    print(f"Running inference on {_test_id} ...")
    _t0      = time.time()
    _raw_out = run_inference(pipe, _prompt,
                             max_new_tokens=MAX_NEW_TOKENS,
                             temperature=TEMPERATURE)
    print(f"Done  |  {time.time() - _t0:.1f}s  |  {len(_raw_out)} chars output")
    print("\n--- Raw output (first 800 chars) ---")
    print(_raw_out[:800])
else:
    print("No article loaded -- skipping inference test.")
    _raw_out = ""


[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Running inference on nDOC_008 ...


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Done  |  60.7s  |  2441 chars output

--- Raw output (first 800 chars) ---
<|channel>thought
Thinking Process:

1.  **Analyze the Request:** The goal is to extract all distinct Fisheries Dispute Events (FDEs) from the provided news article (nDOC_008) and return them as a strict JSON array, adhering to all defined schema and coding rules.

2.  **Analyze the Article (nDOC_008):**
    *   **Topic:** Blast fishing in Tanzania.
    *   **Key Issues:** Illegal use of explosives, destruction of coral reefs, risk to artisanal fishermen, lack of enforcement, high profitability.
    *   **Actors:** Fishermen (illegal/perpetrators), Police, SmartFish/EU (reporting/monitoring).
    *   **Location:** Tanzanian coastline (Mtwara to Tanga, off Zanzibar).
    *   **Time:** Ongoing/Recent (past five years, since 2003 ban).

3.  **Identify Potential FDEs:**
    *   The article des


In [19]:
print(_raw_out)

<|channel>thought
Thinking Process:

1.  **Analyze the Request:** The goal is to extract all distinct Fisheries Dispute Events (FDEs) from the provided news article (nDOC_008) and return them as a strict JSON array, adhering to all defined schema and coding rules.

2.  **Analyze the Article (nDOC_008):**
    *   **Topic:** Blast fishing in Tanzania.
    *   **Key Issues:** Illegal use of explosives, destruction of coral reefs, risk to artisanal fishermen, lack of enforcement, high profitability.
    *   **Actors:** Fishermen (illegal/perpetrators), Police, SmartFish/EU (reporting/monitoring).
    *   **Location:** Tanzanian coastline (Mtwara to Tanga, off Zanzibar).
    *   **Time:** Ongoing/Recent (past five years, since 2003 ban).

3.  **Identify Potential FDEs:**
    *   The article describes a widespread, ongoing *practice* (blast fishing) rather than a single, discrete incident (e.g., a specific arrest on a specific date).
    *   *Event 1:* The general practice of illegal blast f

## 10 · JSON Extractor

Three-layer recovery to parse a JSON array from the model's raw output,
however it chooses to wrap the response:

1. **Markdown-fenced** — strips ` ```json ... ``` ` blocks  
2. **Clean JSON** — parses the whole string directly  
3. **Substring scan** — finds the outermost `[...]` and tries that slice  

Returns the parsed `list`, or `None` on total failure (triggers the
failed-log path in the pipeline).


In [20]:
def extract_json_array(raw: str) -> Optional[list]:
    """
    Pull a JSON array from raw model output using three fallback strategies.
    Returns the parsed list or None if all strategies fail.
    """
    text = raw.strip()

    # Strategy 1 -- strip markdown fences
    if "```" in text:
        parts = text.split("```")
        for part in parts[1::2]:      # content inside fence pairs
            cleaned = part.strip()
            if cleaned.startswith("json"):
                cleaned = cleaned[4:].strip()
            try:
                result = json.loads(cleaned)
                if isinstance(result, list):
                    return result
            except json.JSONDecodeError:
                continue

    # Strategy 2 -- whole string is valid JSON
    try:
        result = json.loads(text)
        if isinstance(result, list):
            return result
    except json.JSONDecodeError:
        pass

    # Strategy 3 -- substring between first [ and last ]
    start, end = text.find("["), text.rfind("]")
    if start != -1 and end > start:
        try:
            result = json.loads(text[start:end + 1])
            if isinstance(result, list):
                return result
        except json.JSONDecodeError:
            pass

    return None


# ── Unit tests ────────────────────────────────────────────────────────────
_arr = '[{"article_id":"X","year":2010}]'
assert extract_json_array(_arr)                             is not None, "clean"
assert extract_json_array(f"```json\n{_arr}\n```")        is not None, "fenced"
assert extract_json_array(f"Here are the events:\n{_arr}") is not None, "buried"
assert extract_json_array("No JSON here.")                  is None,     "none"
print("All extract_json_array tests passed")

# ── Test on actual model output (if available) ────────────────────────────
if _raw_out:
    _parsed = extract_json_array(_raw_out)
    if _parsed is not None:
        print(f"\nParsed {len(_parsed)} event(s) from {_test_id} model output")
    else:
        print("\nCould not parse model output as JSON -- inspect _raw_out above")
else:
    print("\nNo model output to test against yet (run cell 9 first).")


All extract_json_array tests passed

Parsed 0 event(s) from nDOC_008 model output


## 11 · Pydantic Validator

Runs each parsed dict through `FdeEvent` validation:
- **Valid** → clean `model_dump()` dict, ready to write  
- **Invalid** → returned with a `_validation_error` key so it can be
  logged and reviewed without silently discarding the extraction


In [21]:
def validate_events(
    raw_events: list, article_id: str
) -> tuple[list[dict], list[dict]]:
    """
    Validate each event dict through FdeEvent.
    Returns (valid_records, invalid_records).
    Invalid records carry a '_validation_error' key.
    """
    valid, invalid = [], []
    for i, ev in enumerate(raw_events):
        # Inject article_id if the model forgot to include it
        if not ev.get("article_id"):
            ev["article_id"] = article_id
        try:
            fde = FdeEvent(**ev)
            valid.append(fde.model_dump())
        except ValidationError as e:
            log.warning(f"  Validation error on event {i}: {e.error_count()} error(s)")
            ev["_validation_error"]   = str(e)
            ev["_source_article_id"] = article_id
            invalid.append(ev)
    return valid, invalid


# ── Unit tests ────────────────────────────────────────────────────────────
_drivers_false = {k: False for k in ConflictDrivers.model_fields}
_crit_true     = {
    "min_two_actors": True, "discrete_temporal_moment": True,
    "discrete_location": True, "fisheries_resource_contested": True,
}

_good_ev = {
    "article_id": "nDOC_008",
    "fde_criteria": _crit_true,
    "date": "2008-01-01", "year": 2008, "country": "Tanzania",
    "region": 1, "location": "Dar es Salaam", "loc_accuracy": 4,
    "actor1": "Blast fishers", "actor2": "Tanzanian police",
    "ViolenceScore": 2, "drivers": _drivers_false,
    "issue_note": "Police arrested blast fishers. ViolenceScore=2 (arrests).",
}
_bad_ev   = {**_good_ev, "ViolenceScore": 99}   # invalid score
_no_id_ev = {**_good_ev, "article_id": ""}      # ID should be injected

v, inv = validate_events([_good_ev, _bad_ev, _no_id_ev], "INJECT_ID")
assert len(v)   == 2,                       "expected 2 valid"
assert len(inv) == 1,                       "expected 1 invalid"
assert "_validation_error" in inv[0],       "error key present"
assert v[1]["article_id"] == "INJECT_ID",   "article_id injected"
print(f"All validate_events tests passed  |  valid={len(v)}  invalid={len(inv)}")

# ── Test on actual model output (if available) ────────────────────────────
if _raw_out:
    _parsed = extract_json_array(_raw_out)
    if _parsed:
        _valid, _invalid = validate_events(_parsed, _test_id)
        print(f"\nModel output: {len(_valid)} valid, {len(_invalid)} invalid")
        for ev in _valid:
            print(f"  OK  {ev['article_id']} | {ev['date']} | {ev['actor1']} vs {ev['actor2']} | V={ev['ViolenceScore']}")
        for ev in _invalid:
            print(f"  ERR {ev.get('article_id')} -- {ev['_validation_error'][:100]}")


All validate_events tests passed  |  valid=2  invalid=1


## 12 · CSV Row Builder

Flattens a nested `FdeEvent` dict into a single-row dict for the CSV writer.
All nested booleans (`fde_criteria.*`, `drivers.*`) are expanded into
individual columns and cast to `0/1` integers for easy analysis in R/Excel.


In [22]:
DRIVER_FIELDS = [
    "WeakGov", "FishPop", "EcoChngOther",
    "GroundsLim", "ForeignFisher", "IllegalFishing",
    "IncrEfficiency", "OpsScales", "Markets",
    "Poverty", "FoodInsecurity", "Marginalization",
    "StratLoc", "MaritimeCrime"
]

CSV_HEADERS = (
    ["article_id", "date", "year", "country", "region",
     "location", "loc_accuracy", "actor1", "actor2", "ViolenceScore",
     # FDE criteria columns (flattened)
     "fde_min_two_actors", "fde_discrete_temporal_moment",
     "fde_discrete_location", "fde_fisheries_resource_contested"]
    + DRIVER_FIELDS
    + ["issue_note"]
)


def event_to_csv_row(ev: dict) -> dict:
    """Flatten a validated FdeEvent dict into a single CSV row."""
    crit = ev.get("fde_criteria", {})
    drv  = ev.get("drivers", {})
    return {
        "article_id":    ev.get("article_id", ""),
        "date":          ev.get("date", ""),
        "year":          ev.get("year", ""),
        "country":       ev.get("country", ""),
        "region":        ev.get("region", ""),
        "location":      ev.get("location", ""),
        "loc_accuracy":  ev.get("loc_accuracy", ""),
        "actor1":        ev.get("actor1", ""),
        "actor2":        ev.get("actor2", ""),
        "ViolenceScore": ev.get("ViolenceScore", ""),
        "fde_min_two_actors":               int(crit.get("min_two_actors", False)),
        "fde_discrete_temporal_moment":     int(crit.get("discrete_temporal_moment", False)),
        "fde_discrete_location":            int(crit.get("discrete_location", False)),
        "fde_fisheries_resource_contested": int(crit.get("fisheries_resource_contested", False)),
        **{k: int(drv.get(k, False)) for k in DRIVER_FIELDS},
        "issue_note":    ev.get("issue_note", ""),
    }


# ── Test ──────────────────────────────────────────────────────────────────
_row = event_to_csv_row(v[0])   # reuse valid event from cell 11
assert set(_row.keys()) == set(CSV_HEADERS), (
    f"Column mismatch!\n  Got:      {sorted(_row.keys())}\n"
    f"  Expected: {sorted(CSV_HEADERS)}"
)
print(f"CSV row builder OK  |  {len(CSV_HEADERS)} columns")
print("\nSample row preview:")
for col, val in _row.items():
    print(f"  {col:<42} {val}")


CSV row builder OK  |  29 columns

Sample row preview:
  article_id                                 nDOC_008
  date                                       2008-01-01
  year                                       2008
  country                                    Tanzania
  region                                     1
  location                                   Dar es Salaam
  loc_accuracy                               4
  actor1                                     Blast fishers
  actor2                                     Tanzanian police
  ViolenceScore                              2
  fde_min_two_actors                         1
  fde_discrete_temporal_moment               1
  fde_discrete_location                      1
  fde_fisheries_resource_contested           1
  WeakGov                                    0
  FishPop                                    0
  EcoChngOther                               0
  GroundsLim                                 0
  ForeignFisher                   

## 13 · Output Writers

Opens the three output files once at the start of a pipeline run and
provides helpers to write individual records. The files are closed at the
end of cell 14 — do not call `open_output_files()` more than once per run.


In [23]:
def open_output_files(jsonl_path: Path, csv_path: Path, failed_path: Path):
    """
    Open JSONL, CSV, and failed-log files for writing.
    Returns (jsonl_fh, csv_writer, csv_fh, failed_fh).
    """
    jsonl_fh  = open(jsonl_path,  "w", encoding="utf-8")
    failed_fh = open(failed_path, "w", encoding="utf-8")
    csv_fh    = open(csv_path,    "w", newline="", encoding="utf-8")
    csv_writer = csv.DictWriter(csv_fh, fieldnames=CSV_HEADERS)
    csv_writer.writeheader()
    return jsonl_fh, csv_writer, csv_fh, failed_fh


def write_valid_event(jsonl_fh, csv_writer, ev: dict) -> None:
    """Append one validated event to JSONL and CSV."""
    jsonl_fh.write(json.dumps(ev, ensure_ascii=False) + "\n")
    csv_writer.writerow(event_to_csv_row(ev))


def write_failed(failed_fh, article_id: str, error: str, payload) -> None:
    """Append a failure record to the failed log."""
    failed_fh.write(json.dumps({
        "article_id": article_id,
        "error":      error,
        "payload":    payload,
    }) + "\n")


print("Output writer functions defined")
print(f"  Will write to:")
print(f"    JSONL   {JSONL_PATH}")
print(f"    CSV     {CSV_PATH}")
print(f"    Failed  {FAILED_PATH}")


Output writer functions defined
  Will write to:
    JSONL   ../results/fde_events_20260610_200259.jsonl
    CSV     ../results/fde_events_20260610_200259.csv
    Failed  ../results/fde_failed_20260610_200259.jsonl


## 14 · ▶ Full Pipeline Loop

Runs the complete extraction pipeline over all `contains_fde=True` articles.

**All earlier cells must be executed first.**  
Set `LIMIT` in **cell 2** to restrict to the first N articles for testing.


In [24]:
articles_to_process = fde_articles[:LIMIT] if LIMIT else fde_articles
print(f"Processing {len(articles_to_process)} article(s) "
      f"({'limited to ' + str(LIMIT) if LIMIT else 'all FDE-positive'})\n")

# Open output files
jsonl_fh, csv_writer, csv_fh, failed_fh = open_output_files(
    JSONL_PATH, CSV_PATH, FAILED_PATH
)

stats = {
    "total":        len(articles_to_process),
    "processed":    0,
    "events_found": 0,
    "no_events":    0,
    "parse_failed": 0,
    "val_errors":   0,
    "missing_file": 0,
}

for idx, entry in enumerate(articles_to_process, 1):
    article_id = entry["article_id"]
    log.info(f"[{idx}/{stats['total']}] {article_id}")

    # 1. Load article text
    article_text = load_article_text(article_id, ARTICLES_DIR)
    if article_text is None:
        write_failed(failed_fh, article_id, "file_not_found",
                     str(ARTICLES_DIR / f"{article_id}.txt"))
        stats["missing_file"] += 1
        continue

    # 2. Build prompt & run inference
    t0      = time.time()
    prompt  = build_prompt(pipe.tokenizer, article_id, article_text)
    raw     = run_inference(pipe, prompt,
                            max_new_tokens=MAX_NEW_TOKENS,
                            temperature=TEMPERATURE)
    elapsed = time.time() - t0
    log.info(f"  Inference: {elapsed:.1f}s  |  {len(raw)} chars")

    # 3. Parse JSON
    parsed = extract_json_array(raw)
    if parsed is None:
        log.warning("  JSON parse failed -- logged to failed file")
        write_failed(failed_fh, article_id, "json_parse_failed", raw)
        stats["parse_failed"] += 1
        stats["processed"]    += 1
        continue

    if len(parsed) == 0:
        log.info("  Model returned [] -- no FDEs in article")
        stats["no_events"] += 1
        stats["processed"] += 1
        continue

    # 4. Validate
    valid_events, invalid_events = validate_events(parsed, article_id)

    for ev in invalid_events:
        log.warning("  Validation error -- logged to failed file")
        write_failed(failed_fh, article_id, "validation_error", ev)
        stats["val_errors"] += 1

    # 5. Write valid events
    for ev in valid_events:
        write_valid_event(jsonl_fh, csv_writer, ev)

    stats["events_found"] += len(valid_events)
    stats["processed"]    += 1
    log.info(f"  {len(valid_events)} valid, {len(invalid_events)} invalid  "
             f"| running total: {stats['events_found']}")

# Close all files
jsonl_fh.close()
failed_fh.close()
csv_fh.close()

# Summary
print("\n" + "=" * 55)
print("PIPELINE COMPLETE")
print("=" * 55)
print(f"  Articles processed       : {stats['processed']} / {stats['total']}")
print(f"  Valid FDE events found   : {stats['events_found']}")
print(f"  Articles -> no events    : {stats['no_events']}")
print(f"  JSON parse failures      : {stats['parse_failed']}")
print(f"  Pydantic validation errs : {stats['val_errors']}")
print(f"  Missing article files    : {stats['missing_file']}")
print(f"  Output JSONL             : {JSONL_PATH}")
print(f"  Output CSV               : {CSV_PATH}")
print(f"  Failed log               : {FAILED_PATH}")
print("=" * 55)


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing 2 article(s) (limited to 3)



[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PIPELINE COMPLETE
  Articles processed       : 2 / 2
  Valid FDE events found   : 1
  Articles -> no events    : 1
  JSON parse failures      : 0
  Pydantic validation errs : 0
  Missing article files    : 0
  Output JSONL             : ../results/fde_events_20260610_200259.jsonl
  Output CSV               : ../results/fde_events_20260610_200259.csv
  Failed log               : ../results/fde_failed_20260610_200259.jsonl


In [25]:
raw

'<|channel>thought\nThinking Process:\n\n1.  **Analyze the Request:** The goal is to act as an expert coder for the Fisheries Conflict Database, reading the provided news article (nDOC_012) and extracting all distinct Fisheries Dispute Events (FDEs). The output must be a strict JSON array, with no markdown fences or commentary.\n\n2.  **Analyze the Article (nDOC_012):**\n    *   **Topic:** Tanzania confronting pirate fishing in its EEZ.\n    *   **Key Incident:** Arrest of a flagless foreign trawler (\'Tawaqal 1\') 100 miles off the Tanzanian coast.\n    *   **Date/Time:** "Last week\'s arrest" (relative to March 15, 2009). The ship was towed "last Monday."\n    *   **Location:** 100 miles off the Tanzanian coast / Indian Ocean waters / Dar es Salaam harbour.\n    *   **Actors:** Tanzania Government/Police (responding), Foreign Pirate Fishing Vessels (initiating/primary party), South African-led joint operation (cooperating).\n    *   **Conflict:** Illegal, Unreported, Unregulated (IUU

## 15 · Results Inspector

Re-reads the output JSONL and prints a human-readable event summary.
Run after cell 14 to review results without opening files manually.


In [26]:
events_out = []
if JSONL_PATH.exists():
    with open(JSONL_PATH) as f:
        events_out = [json.loads(l) for l in f if l.strip()]

print(f"Events in output file: {len(events_out)}\n")

for i, ev in enumerate(events_out, 1):
    on_drivers = [k for k, val in ev.get("drivers", {}).items() if val]
    print(f"Event #{i}  [{ev['article_id']}]")
    print(f"  Date      : {ev['date']}  (year {ev['year']})")
    print(f"  Country   : {ev['country']}  |  Location : {ev['location']}  [acc={ev['loc_accuracy']}]")
    print(f"  Region    : {ev['region']}")
    print(f"  Actor 1   : {ev['actor1']}")
    print(f"  Actor 2   : {ev['actor2']}")
    print(f"  Violence  : {ev['ViolenceScore']}")
    print(f"  Drivers   : {', '.join(on_drivers) if on_drivers else 'none'}")
    print(f"  Note      : {ev['issue_note'][:160]}")
    print()

# Also show failed-log summary if any
failed_out = []
if FAILED_PATH.exists():
    with open(FAILED_PATH) as f:
        failed_out = [json.loads(l) for l in f if l.strip()]

if failed_out:
    print(f"Failed records: {len(failed_out)}")
    for rec in failed_out:
        print(f"  {rec['article_id']}  |  error: {rec['error']}")
else:
    print("No failed records.")


Events in output file: 1

Event #1  [nDOC_012]
  Date      : 2009-03-08  (year 2009)
  Country   : Tanzania  |  Location : 100 miles off the Tanzanian coast  [acc=5]
  Region    : 1
  Actor 1   : Foreign Pirate Fishing Vessels
  Actor 2   : Tanzania Government/Police
  Violence  : 2
  Drivers   : WeakGov, GroundsLim, ForeignFisher, IllegalFishing, Markets
  Note      : The FDE involves the seizure of the flagless trawler 'Tawaqal 1' by Tanzanian authorities. The ViolenceScore is 2 due to the physical action of arrest and confi

No failed records.
